In [ ]:
#!/usr/bin/env python
# coding: utf-8
import sys
sys.path.append("../../pipeline/")
sys.path.append("../../")

sys.path.append("../../experiments/parametrization_experiments/")

sys.path.append('../../'); sys.path.append('../../../'); sys.path.append('../../../gmsh'); sys.path.append('../../Visualization/')


sys.path.append('periodic_patches/')
sys.path.append('gmsh')
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf
from tri_mesh_viewer import TriMeshViewer

import parallelism
parallelism.set_max_num_tbb_threads(4)
import parametrization_experiment_helper
import os 
import inflation
from mesh_utilities import SurfaceSampler, tubeRemesh
import boundaries
import time 
import parametrization_helper
import json 
import sheet_optimizer, opt_config

data_time_stamp = 'meshing_output_low_res_2024_01_18_23_29'

# output_time_stamp = time.strftime("%Y_%m_%d_%H_%M")
output_time_stamp = 'low_res_2024_01_18_23_29'

In [ ]:
import sheet_optimizer

### Igloo

In [ ]:
tag = 'free_boundary'

In [ ]:
# shape_name = 'squiward'
# pattern_name = 'dash_line'

shape_name = parametrization_experiment_helper.Shape_data[1]['name']
pattern_name = parametrization_experiment_helper.Pattern_data[1]['name']



In [ ]:
meshing_data_path = '../../pipeline/inverse_design/output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
optimization_data_path = '../../pipeline/inverse_design/output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)

In [ ]:
# print("Load existing optimization for tag {}".format(tag))
# sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_19_23_48_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
# targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

print("Load existing optimization for tag {}".format(tag))
sheet_opt = sheet_optimizer.load(optimization_data_path + '{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

In [ ]:

tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
targetSurf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)


In [ ]:
from visualization import TriMeshViewerWithSurface

In [ ]:
nv = tas.sheet().mesh().numVertices()

In [ ]:
isheet = tas.sheet()

In [ ]:
iwv = [isheet.isWallVtx(i) for i in range(isheet.mesh().numVertices())]


In [ ]:
flat_sheet = inflation.InflatableSheet(tas.sheet().mesh(), iwv)
flat_vars = flat_sheet.getVars().reshape((-1, 3))
flat_centroid = np.mean(flat_vars, axis = 0)

In [ ]:
target_vars = tas.getVars().reshape((-1, 3))
target_centroid = np.mean(target_vars, axis = 0)

In [ ]:
target_centroid

In [ ]:
new_vars = (flat_vars - flat_centroid + target_centroid).flatten()

In [ ]:
tas.sheet().setVars(new_vars)

In [ ]:
viewer = TriMeshViewerWithSurface(tas, targetSurf, width=768, height=640)
viewer.showWireframe(False)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))


def cb(it):
    viewer.update()



In [ ]:
viewer.show()

In [ ]:
camParams = viewer.getCameraParams()

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(tas, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(camParams)

In [ ]:
viewer.recordStart("igloo.mp4")

In [ ]:
# fixedvars = boundaries.getOuterBoundaryVars(isheet)
fixedvars = []

In [ ]:
tas.fittingWeight = 1e-2
sheet_opt.opts.niter = 5

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-3
sheet_opt.opts.niter = 10

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-8
sheet_opt.opts.niter = 500

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)


In [ ]:
viewer.recordStop()

### hill

In [ ]:
tag = 'fixed_boundary'

In [ ]:
# shape_name = 'squiward'
# pattern_name = 'dash_line'

shape_name = parametrization_experiment_helper.Shape_data[0]['name']
pattern_name = parametrization_experiment_helper.Pattern_data[2]['name']



In [ ]:
meshing_data_path = '../../pipeline/inverse_design/output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
optimization_data_path = '../../pipeline/inverse_design/output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)

In [ ]:
# print("Load existing optimization for tag {}".format(tag))
# sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_19_23_48_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
# targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

print("Load existing optimization for tag {}".format(tag))
sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_22_17_20_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

In [ ]:

tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
target_surf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)


In [ ]:
from visualization import TriMeshViewerWithSurface

In [ ]:
uv = np.load(meshing_data_path + '/rparam_uv.npy')
paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(tas.sheet().mesh().vertices(), target_surf.vertices())


In [ ]:
tas.sheet().setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
viewer = TriMeshViewerWithSurface(tas, target_surf, width=1000, height=1000)
viewer.showWireframe(False)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))


def cb(it):
    viewer.update()



In [ ]:
viewer.show()

In [ ]:
camParams = viewer.getCameraParams()

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(tas, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(camParams)

In [ ]:
viewer.recordStart("hills.mp4")

In [ ]:
fixedvars = boundaries.getOuterBoundaryVars(isheet)

In [ ]:
tas.fittingWeight = 1e-2
sheet_opt.opts.niter = 5

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-3
sheet_opt.opts.niter = 10

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-8
sheet_opt.opts.niter = 10

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)


In [ ]:
viewer.recordStop()

### Cashew

In [ ]:
tag = 'fixed_boundary'

In [ ]:
# shape_name = 'squiward'
# pattern_name = 'dash_line'

shape_name = parametrization_experiment_helper.Shape_data[4]['name']
pattern_name = parametrization_experiment_helper.Pattern_data[1]['name']



In [ ]:
meshing_data_path = '../../pipeline/inverse_design/output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
optimization_data_path = '../../pipeline/inverse_design/output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)

In [ ]:
# print("Load existing optimization for tag {}".format(tag))
# sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_19_23_48_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
# targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

print("Load existing optimization for tag {}".format(tag))
sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_21_01_43_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

In [ ]:

tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
target_surf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)


In [ ]:
from visualization import TriMeshViewerWithSurface

In [ ]:
uv = np.load(meshing_data_path + '/rparam_uv.npy')
paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(tas.sheet().mesh().vertices(), target_surf.vertices())


In [ ]:
tas.sheet().setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
viewer = TriMeshViewerWithSurface(tas, target_surf, width=1000, height=1000)
viewer.showWireframe(False)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))


def cb(it):
    viewer.update()



In [ ]:
viewer.show()

In [ ]:
camParams = viewer.getCameraParams()

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(tas, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(camParams)

In [ ]:
viewer.recordStart("cashew.mp4")

In [ ]:
fixedvars = boundaries.getOuterBoundaryVars(tas.sheet())

In [ ]:
tas.fittingWeight = 1e-2
sheet_opt.opts.niter = 5

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-3
sheet_opt.opts.niter = 10

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-8
sheet_opt.opts.niter = 500

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)


In [ ]:
viewer.recordStop()

### Neck brace

In [ ]:
tag = 'fixed_boundary'

In [ ]:
# shape_name = 'squiward'
# pattern_name = 'dash_line'

shape_name = parametrization_experiment_helper.Shape_data[3]['name']
pattern_name = parametrization_experiment_helper.Pattern_data[2]['name']



In [ ]:
meshing_data_path = '../../pipeline/inverse_design/output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
optimization_data_path = '../../pipeline/inverse_design/output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)

In [ ]:
# print("Load existing optimization for tag {}".format(tag))
# sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_19_23_48_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
# targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

print("Load existing optimization for tag {}".format(tag))
sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_22_17_00_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

In [ ]:

tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
target_surf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)


In [ ]:
from visualization import TriMeshViewerWithSurface

In [ ]:
uv = np.load(meshing_data_path + '/rparam_uv.npy')
paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(tas.sheet().mesh().vertices(), target_surf.vertices())


In [ ]:
tas.sheet().setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
viewer = TriMeshViewerWithSurface(tas, target_surf, width=768, height=640)
viewer.showWireframe(False)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))


def cb(it):
    viewer.update()



In [ ]:
viewer.show()

In [ ]:
camParams = viewer.getCameraParams()

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(tas, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(camParams)

In [ ]:
viewer.recordStart("neck_brace.mp4")

In [ ]:
fixedvars = boundaries.getOuterBoundaryVars(tas.sheet())

In [ ]:
tas.fittingWeight = 1e-2
sheet_opt.opts.niter = 5

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-3
sheet_opt.opts.niter = 10

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-8
sheet_opt.opts.niter = 500

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)


In [ ]:
viewer.recordStop()

### Squidward

In [ ]:
import matplotlib

In [ ]:
tag = 'fixed_boundary'

In [ ]:
# shape_name = 'squiward'
# pattern_name = 'dash_line'

shape_name = parametrization_experiment_helper.Shape_data[5]['name']
pattern_name = parametrization_experiment_helper.Pattern_data[0]['name']



In [ ]:
meshing_data_path = '../../pipeline/inverse_design/output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
optimization_data_path = '../../pipeline/inverse_design/output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)

In [ ]:
# print("Load existing optimization for tag {}".format(tag))
# sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_19_23_48_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
# targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

print("Load existing optimization for tag {}".format(tag))
sheet_opt = sheet_optimizer.load(optimization_data_path + '{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

In [ ]:

tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
target_surf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)


In [ ]:
from visualization import TriMeshViewerWithSurface

In [ ]:
uv = np.load(meshing_data_path + '/rparam_uv.npy')
paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(tas.sheet().mesh().vertices(), target_surf.vertices())


In [ ]:
tas.sheet().setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
viewer = TriMeshViewerWithSurface(tas, target_surf, width=768, height=640)
viewer.showWireframe(False)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))


def cb(it):
    viewer.update()



In [ ]:
viewer.show()

In [ ]:
camParams = viewer.getCameraParams()

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(tas, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(camParams)

In [ ]:
viewer.recordStart("squidward.mp4")

In [ ]:
fixedvars = boundaries.getOuterBoundaryVars(tas.sheet())
# fixedvars = []

In [ ]:
tas.fittingWeight = 1e-2
sheet_opt.opts.niter = 5

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-3
sheet_opt.opts.niter = 10

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-8
sheet_opt.opts.niter = 500

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)


In [ ]:
viewer.recordStop()

### lemonade

In [ ]:
tag = 'fixed_boundary'

In [ ]:
# shape_name = 'squiward'
# pattern_name = 'dash_line'

shape_name = parametrization_experiment_helper.Shape_data[7]['name']
pattern_name = parametrization_experiment_helper.Pattern_data[1]['name']



In [ ]:
meshing_data_path = '../../pipeline/inverse_design/output/{}/{}_{}/'.format(data_time_stamp, shape_name, pattern_name)
optimization_data_path = '../../pipeline/inverse_design/output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)

In [ ]:
# print("Load existing optimization for tag {}".format(tag))
# sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_19_23_48_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
# targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

print("Load existing optimization for tag {}".format(tag))
sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_21_16_15_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

In [ ]:

tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
target_surf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)


In [ ]:
from visualization import TriMeshViewerWithSurface

In [ ]:
uv = np.load(meshing_data_path + '/rparam_uv.npy')
paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(tas.sheet().mesh().vertices(), target_surf.vertices())


In [ ]:
tas.sheet().setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
viewer = TriMeshViewerWithSurface(tas, target_surf, width=1000, height=1000)
viewer.showWireframe(False)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))


def cb(it):
    viewer.update()



In [ ]:
viewer.show()

In [ ]:
camParams = viewer.getCameraParams()

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(tas, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(camParams)

In [ ]:
viewer.recordStart("lemonade.mp4")

In [ ]:
isheet = tas.sheet()

In [ ]:
print("Fix feet!")
m = isheet.mesh()
V = m.vertices()
BV = m.boundaryVertices()
arclen = lambda l: np.linalg.norm(np.diff(V[BV[np.array(l)]], axis=0), axis=1).sum()
outerLoopBdryVertices = max(m.boundaryLoops(), key=arclen)
feet_fixed_vars = []
for bvi in outerLoopBdryVertices:
    for c in range(3):
        feet_fixed_vars.append(isheet.varIdx(0, BV[bvi], c))

feet_fixed_vars = np.array(feet_fixed_vars).reshape((-1, 3))

sheet_vars = isheet.getVars()

fixed_vars_values = sheet_vars[feet_fixed_vars[:, 1]]

bottom_fixed_vars = []
for i in range(len(feet_fixed_vars)):
    if np.abs(fixed_vars_values[i] - min(fixed_vars_values))< 1:
        bottom_fixed_vars.extend(feet_fixed_vars[i])
bdryVars = np.array(bottom_fixed_vars)
print(bdryVars)

fixedvars = bdryVars

In [ ]:
tas.fittingWeight = 1e-2
sheet_opt.opts.niter = 5

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-3
sheet_opt.opts.niter = 10

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

tas.fittingWeight = 1e-8
sheet_opt.opts.niter = 100

# Remove the target-attraction force and recompute the equilibrium
inflation.inflation_newton(tas, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)


In [ ]:
viewer.recordStop()